In [4]:
import json
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from ipywidgets import interact, IntSlider, Checkbox, VBox, HBox, Layout, Output
from IPython.display import display, HTML, clear_output

In [5]:
PRED_JSON_PATH = r"../outputs/round3_20_35_pred.json"

with open(PRED_JSON_PATH, "r", encoding="utf-8") as f:
    pred_data = json.load(f)

meta = pred_data.get("meta", {})
results = pred_data.get("results", [])

print("Loaded file:", PRED_JSON_PATH)
print("Num frames:", len(results))
print("Meta:", meta)

Loaded file: ../outputs/round3_20_35_pred.json
Num frames: 121
Meta: {'json_path': 'examples/PGL-vit-pv-inferno.json', 'round_id': 18, 'start_sec': 70.0, 'end_sec': 100.0, 'tick_stride': 1, 'save_mode': 'attach', 'remove_projectiles': False, 'num_ticks': 121}


In [6]:
def get_prediction_block(frame_item):
    """
    兼容两种保存格式：
    1) attach: frame_item["model_prediction"]
    2) prediction_only: frame_item["tick_prediction"]
    """
    if "model_prediction" in frame_item:
        return frame_item["model_prediction"]
    elif "tick_prediction" in frame_item:
        return frame_item["tick_prediction"]
    else:
        return {}


def get_players_info(frame_item):
    """
    attach 模式下，原始 tick 会保留 players_info
    """
    return frame_item.get("players_info", [])


def yaw_to_unit_vector(yaw_deg):
    """
    将 yaw 角度转成 2D 朝向向量
    尽量贴近你原始 ASCII 版的 4 向感知，但这里用连续方向更直观
    """
    rad = math.radians(yaw_deg)
    dx = math.cos(rad)
    dy = math.sin(rad)
    return dx, dy


def normalize_positions(players_info, padding=0.08):
    xs = [p["X"] for p in players_info]
    ys = [p["Y"] for p in players_info]

    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    if abs(max_x - min_x) < 1e-6:
        max_x += 1.0
    if abs(max_y - min_y) < 1e-6:
        max_y += 1.0

    span_x = max_x - min_x
    span_y = max_y - min_y

    min_x -= padding * span_x
    max_x += padding * span_x
    min_y -= padding * span_y
    max_y += padding * span_y

    return min_x, max_x, min_y, max_y


def format_prob(x, digits=4):
    if x is None:
        return "-"
    return f"{float(x):.{digits}f}"


def build_player_table_html(players_info, pred):
    """
    右侧表格：每人状态 + 模型预测
    """
    player_preds = pred.get("player_predictions", [])

    rows = []
    for i, p in enumerate(players_info):
        pred_i = player_preds[i] if i < len(player_preds) else {}
        rows.append(f"""
        <tr>
            <td>{i}</td>
            <td>{p.get('name', f'player_{i}')}</td>
            <td>{p.get('team_num', '-')}</td>
            <td>{'Alive' if p.get('is_alive', False) else 'Dead'}</td>
            <td>{format_prob(pred_i.get('alive_prob'))}</td>
            <td>{format_prob(pred_i.get('next_killer_prob'))}</td>
            <td>{format_prob(pred_i.get('next_death_prob'))}</td>
        </tr>
        """)

    html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px;">
      <h4 style="margin-bottom:6px;">Player Predictions</h4>
      <table style="border-collapse: collapse; width: 100%;">
        <thead>
          <tr style="background:#f0f0f0;">
            <th style="border:1px solid #ccc; padding:4px;">Idx</th>
            <th style="border:1px solid #ccc; padding:4px;">Name</th>
            <th style="border:1px solid #ccc; padding:4px;">Team</th>
            <th style="border:1px solid #ccc; padding:4px;">GT Alive</th>
            <th style="border:1px solid #ccc; padding:4px;">alive_prob</th>
            <th style="border:1px solid #ccc; padding:4px;">next_killer_prob</th>
            <th style="border:1px solid #ccc; padding:4px;">next_death_prob</th>
          </tr>
        </thead>
        <tbody>
          {''.join(rows)}
        </tbody>
      </table>
    </div>
    """
    return html


def build_summary_html(frame_item, pred):
    ct_win_rate = pred.get("ct_win_rate", None)
    round_id = frame_item.get("round", "-")
    round_sec = frame_item.get("round_seconds", "-")

    nk = pred.get("next_killer_probs", {})
    nd = pred.get("next_death_probs", {})

    html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 14px; margin-bottom:10px;">
      <h3 style="margin:0 0 8px 0;">Frame Summary</h3>
      <p style="margin:4px 0;"><b>Round:</b> {round_id}</p>
      <p style="margin:4px 0;"><b>Round Seconds:</b> {round_sec}</p>
      <p style="margin:4px 0;"><b>CT Win Rate:</b> {format_prob(ct_win_rate)}</p>
      <p style="margin:4px 0;"><b>No Kill Prob:</b> {format_prob(nk.get('no_kill'))}</p>
      <p style="margin:4px 0;"><b>No Death Prob:</b> {format_prob(nd.get('no_death'))}</p>
    </div>
    """
    return html


def show_duel_matrix(pred, max_name_len=10):
    duel = pred.get("duel_matrix", None)
    player_preds = pred.get("player_predictions", [])
    if duel is None or len(player_preds) == 0:
        print("No duel matrix available.")
        return

    names = [p.get("name", f"p{i}")[:max_name_len] for i, p in enumerate(player_preds)]
    duel_np = np.array(duel)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(duel_np, aspect="auto")

    ax.set_xticks(np.arange(len(names)))
    ax.set_yticks(np.arange(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_yticklabels(names)

    ax.set_title("Duel Matrix")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

In [7]:
out_info = Output()
out_table = Output()
out_duel = Output()

def render_frame(frame_idx=0, show_names=True, show_arrows=True, show_duel=False):
    frame_item = results[frame_idx]
    pred = get_prediction_block(frame_item)
    players_info = get_players_info(frame_item)

    if len(players_info) == 0:
        print("当前结果不包含 players_info。请确认你保存时使用的是 save_mode=attach。")
        return

    min_x, max_x, min_y, max_y = normalize_positions(players_info)

    fig, ax = plt.subplots(figsize=(8, 6))

    # 分队绘制
    for i, p in enumerate(players_info):
        x, y = p["X"], p["Y"]
        team = p.get("team_num", "UNK")
        is_alive = p.get("is_alive", False)
        yaw = p.get("yaw", 0.0)
        name = p.get("name", f"player_{i}")

        if not is_alive:
            color = "gray"
            alpha = 0.5
            marker = "x"
            size = 90
        else:
            if team == "CT":
                color = "tab:blue"
            else:
                color = "goldenrod"
            alpha = 0.95
            marker = "o"
            size = 140

        ax.scatter(x, y, c=color, s=size, marker=marker, alpha=alpha, edgecolors="black", linewidths=0.8)

        # 方向箭头
        if show_arrows and is_alive:
            dx, dy = yaw_to_unit_vector(yaw)
            arrow_len = 0.03 * max(max_x - min_x, max_y - min_y)
            arrow = FancyArrowPatch(
                (x, y),
                (x + dx * arrow_len, y + dy * arrow_len),
                arrowstyle='->',
                mutation_scale=12,
                linewidth=1.5,
                color=color
            )
            ax.add_patch(arrow)

        # 名字 + alive_prob
        if show_names:
            alive_prob = None
            player_preds = pred.get("player_predictions", [])
            if i < len(player_preds):
                alive_prob = player_preds[i].get("alive_prob", None)

            label = f"{name}\nA:{format_prob(alive_prob, 3)}"
            ax.text(
                x, y,
                label,
                fontsize=8,
                ha="center",
                va="bottom",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.75, edgecolor="none")
            )

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.set_title(f"Frame {frame_idx} | Round {frame_item.get('round', '-')} | t={frame_item.get('round_seconds', '-')}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()
    plt.show()

    with out_info:
        clear_output(wait=True)
        display(HTML(build_summary_html(frame_item, pred)))

    with out_table:
        clear_output(wait=True)
        display(HTML(build_player_table_html(players_info, pred)))

    with out_duel:
        clear_output(wait=True)
        if show_duel:
            show_duel_matrix(pred)

In [ ]:
frame_slider = IntSlider(
    value=0,
    min=0,
    max=max(len(results) - 1, 0),
    step=1,
    description="Frame",
    continuous_update=False,
    layout=Layout(width="800px")
)

ui = interact(
    render_frame,
    frame_idx=frame_slider,
    show_names=Checkbox(value=True, description="Show Names"),
    show_arrows=Checkbox(value=True, description="Show Arrows"),
    show_duel=Checkbox(value=False, description="Show Duel Matrix")
)

display(VBox([
    out_info,
    out_table,
    out_duel
]))

interactive(children=(IntSlider(value=0, continuous_update=False, description='Frame', layout=Layout(width='80…